In [2]:
# %pip install optuna

In [23]:
import optuna
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np


In [6]:
# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

In [7]:
df = pd.read_csv(url, names=columns)

In [8]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [21]:
df.Insulin.value_counts() # it has missing values inform of 0

Insulin
0      374
105     11
130      9
140      9
120      8
      ... 
178      1
127      1
510      1
16       1
112      1
Name: count, Length: 186, dtype: int64

In [24]:
# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [25]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


In [26]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize

In [27]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2025-12-03 17:44:28,056] A new study created in memory with name: no-name-517851fe-3a57-4a84-b78f-72a0fad07a5c
[I 2025-12-03 17:44:30,015] Trial 0 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 182, 'max_depth': 8}. Best is trial 0 with value: 0.7616387337057727.
[I 2025-12-03 17:44:31,225] Trial 1 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 135, 'max_depth': 11}. Best is trial 1 with value: 0.7690875232774674.
[I 2025-12-03 17:44:32,039] Trial 2 finished with value: 0.7560521415270017 and parameters: {'n_estimators': 103, 'max_depth': 3}. Best is trial 1 with value: 0.7690875232774674.
[I 2025-12-03 17:44:33,694] Trial 3 finished with value: 0.756052141527002 and parameters: {'n_estimators': 189, 'max_depth': 9}. Best is trial 1 with value: 0.7690875232774674.
[I 2025-12-03 17:44:35,165] Trial 4 finished with value: 0.7765363128491619 and parameters: {'n_estimators': 169, 'max_depth': 7}. Best is trial 4 with value: 0.776536312

In [28]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7821229050279329
Best hyperparameters: {'n_estimators': 115, 'max_depth': 15}


In [30]:
# now training our model with final parameter values

from sklearn.metrics import accuracy_score

best_model = RandomForestClassifier(**study.best_trial.params, random_state=42) #best hyperparameters from Optuna
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)

print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.75


### Implementing grid search cv using optuna

In [31]:
# we need to specifically specify search space
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [34]:
# Define the objective function
def objective(trial):
    
    # Even though n_estimators is an integer, for GridSampler we use suggest_categorical.
    # This allows us to pass the specific list [50, 100, 200] directly.
    n_estimators = trial.suggest_categorical('n_estimators', search_space['n_estimators'])
    max_depth = trial.suggest_categorical('max_depth', search_space['max_depth'])

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy', n_jobs = -1).mean()
    return score

In [35]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective) # now it will not use baysian optimaization

[I 2025-12-03 18:18:19,016] A new study created in memory with name: no-name-588eab48-5eec-41e7-84cc-10b758648a47
[I 2025-12-03 18:18:29,059] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2025-12-03 18:18:32,317] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2025-12-03 18:18:32,570] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2025-12-03 18:18:32,997] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2025-12-03 18:18:33,485] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811